# Domain pilot: scientific and forum texts in the distillation mix

**Why.** The voyage pilot showed that a teacher's domain strength does not reach the student through our mix: the voyage-only student imitated voyage well (cosine 0.83) but got SciFact 69.9 against the teacher's 86.4 (BGE-only student 72.1). The mix has almost no scientific or forum Q&A text. This mix (`data/mix/mix_domain_250k.jsonl`, built by `turkish_distill.domain_mix`) adds 50,000 native Turkish texts to the 200K mix:
- thesis abstracts (25K) and thesis titles (5K) from the YÖK theses dataset (CC-BY-4.0), by subject group: life/health sciences 50% (SciFact, NFCorpus), economics/finance/banking 20% (FiQA; no native Turkish finance corpus exists), other fields 30% (SCIDOCS);
- tech and gaming forum posts (10K question-like posts containing "?", 10K other posts) from ForumSohbetleri (Technopat, DonanımHaber, Wardom, DonanımArşivi; CC-BY-SA-4.0), signatures stripped.
All 13-gram decontaminated against TR-MTEB + Mizan + the RAG sets. Layout: 200K mix first 198K + new texts + last 2K, so the existing caches are copied and only the new texts are encoded.

**Runs** (lr 1e-4, 5400 steps, seed 42; only the corpus differs from the references):
1. `domain_two_s5400`: BGE-M3 + plain 8K-embed. Reference: the plain pair on the 200K mix, 67.16 (seed 43: 67.25); SciFact 72.5, CQADupstack 56.5, SCIDOCS 3.3. Question: do domain texts move the target tasks with the current teachers (BGE teacher SciFact 78.8)?
2. `domain_voyage_single_s5400`: voyage-4-nano alone. Reference: the voyage-only student on the 200K mix, 63.97, SciFact 69.9. Question: does voyage's SciFact strength (86.4) transfer once the texts are there?

Read SciFact / SCIDOCS / CQADupstack / FiQA / ArguAna and the overall mean; differences above 0.3 only. These texts were chosen for these tasks, so a gain must also be checked on Mizan and reported as targeted data.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)


In [ ]:
# --- 1. Encode the new texts (copies the 200K mix's shards for all three plain teachers, encodes the rest) ---
V2 = '/content/drive/MyDrive/bitirme/data/mix/mix_v2_200k.jsonl'
DOMAIN = '/content/drive/MyDrive/bitirme/data/mix/mix_domain_250k.jsonl'
assert os.path.exists(DOMAIN), 'upload data/mix/mix_domain_250k.jsonl to Drive first'
!python -m turkish_distill.encode_teachers --config configs/three_bge_8k_voyage.yaml --override data.corpus_path={DOMAIN} --seed-from-corpus {V2}


In [ ]:
# --- 2. Training runs ---
# Runs of this notebook, one after another. (name, config, extra overrides). A later override wins.
DOMAIN = '/content/drive/MyDrive/bitirme/data/mix/mix_domain_250k.jsonl'
BASE = [f'data.corpus_path={DOMAIN}', 'train.max_steps=5400', 'train.lr=0.0001']
RUNS = [
    ('domain_two_s5400', 'configs/stage1_two_bge_m3_modernbert_tr_embed.yaml', []),   # BGE-M3 + plain 8K-embed
    ('domain_voyage_single_s5400', 'configs/single_voyage.yaml', []),
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        run = os.path.basename(os.path.dirname(d))
        for suffix in ('', '_ema', '_swa'):
            if os.path.isdir(f'{DRIVE_ROOT}/runs/{name}/{run}/final{suffix}'):
                print(f"    'd_{name}{suffix}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{run}/final{suffix}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()